In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
print("✅ Libraries loaded")

✅ Libraries loaded


In [3]:
import os
print(os.getcwd())

# Change path based on your getcwd output
nav_df  = pd.read_csv("data/processed/nav_history_cleaned.csv")
fund_df = pd.read_csv("data/processed/fund_master_cleaned.csv")
nav_df["date"] = pd.to_datetime(nav_df["date"])
nav_df = nav_df.sort_values(["amfi_code", "date"])
nav_df["daily_return"] = nav_df.groupby("amfi_code")["nav"].pct_change()
nav_df = nav_df.dropna(subset=["daily_return"])

print("✅ Data loaded")
print("NAV shape:", nav_df.shape)

C:\Users\hp\mutual_fund_analytics
✅ Data loaded
NAV shape: (45922, 4)


In [4]:
def calculate_cagr(df, amfi_code):
    fund = df[df["amfi_code"] == amfi_code].sort_values("date")
    if len(fund) < 2:
        return None
    start_nav = fund.iloc[0]["nav"]
    end_nav   = fund.iloc[-1]["nav"]
    n_days    = (fund.iloc[-1]["date"] - fund.iloc[0]["date"]).days
    n_years   = n_days / 365
    if n_years == 0:
        return None
    cagr = ((end_nav / start_nav) ** (1 / n_years) - 1) * 100
    return round(cagr, 2)

cagr_results = []
for code in nav_df["amfi_code"].unique():
    cagr = calculate_cagr(nav_df, code)
    name = fund_df[fund_df["amfi_code"] == code]["scheme_name"].values
    if len(name) > 0 and cagr is not None:
        cagr_results.append({
            "amfi_code": code,
            "scheme_name": name[0],
            "cagr_pct": cagr
        })

cagr_df = pd.DataFrame(cagr_results).sort_values(
    "cagr_pct", ascending=False
)
print("=== CAGR Results ===")
print(cagr_df)
cagr_df.to_csv("../data/processed/cagr_results.csv", index=False)
print("✅ CAGR saved")

=== CAGR Results ===
    amfi_code                                        scheme_name  cagr_pct
0      100016          HDFC Top 100 Fund - Regular Plan - Growth       NaN
1      100025       HDFC Short Term Debt Fund - Regular - Growth       NaN
2      100033  HDFC Mid-Cap Opportunities Fund - Regular - Gr...       NaN
3      101206      ABSL Frontline Equity Fund - Regular - Growth       NaN
4      101207             ABSL Small Cap Fund - Regular - Growth       NaN
5      101208                ABSL Liquid Fund - Regular - Growth       NaN
6      102885         UTI Nifty 50 Index Fund - Regular - Growth       NaN
7      102886                UTI Mid Cap Fund - Regular - Growth       NaN
8      102887              UTI Flexi Cap Fund - Regular - Growth       NaN
9      118632     Nippon India Large Cap Fund - Regular - Growth       NaN
10     118633      Nippon India Large Cap Fund - Direct - Growth       NaN
11     118634     Nippon India Small Cap Fund - Regular - Growth       NaN
12  

OSError: Cannot save file into a non-existent directory: '..\data\processed'